In [25]:
import pandas as pd

X_train = pd.read_parquet('../data/X_train.parquet')
X_test = pd.read_parquet('../data/X_test.parquet')
y_train = pd.read_parquet('../data/y_train.parquet')['churn']
y_test = pd.read_parquet('../data/y_test.parquet')['churn']

X_train.shape, X_test.shape

((8000, 12), (2000, 12))

In [26]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

d:\CustomerIQ\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
d:\CustomerIQ\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=1000)

In [27]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [28]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

d:\CustomerIQ\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


LogisticRegression(max_iter=1000)

In [29]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test_scaled)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.82      0.97      0.89      1593
           1       0.58      0.19      0.28       407

    accuracy                           0.81      2000
   macro avg       0.70      0.58      0.59      2000
weighted avg       0.77      0.81      0.77      2000



In [30]:
model_balanced = LogisticRegression(max_iter=1000, class_weight='balanced')
model_balanced.fit(X_train_scaled, y_train)

y_pred_balanced = model_balanced.predict(X_test_scaled)
print(classification_report(y_test, y_pred_balanced))

              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1593
           1       0.39      0.71      0.50       407

    accuracy                           0.71      2000
   macro avg       0.65      0.71      0.65      2000
weighted avg       0.80      0.71      0.74      2000



d:\CustomerIQ\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


In [31]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

rf_model = RandomForestClassifier(random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.87      0.97      0.92      1593
           1       0.80      0.45      0.57       407

    accuracy                           0.86      2000
   macro avg       0.84      0.71      0.75      2000
weighted avg       0.86      0.86      0.85      2000



In [32]:
import pandas as pd

importances = pd.Series(rf_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
importances

age                 0.250673
estimated_salary    0.137645
products_number     0.134518
credit_score        0.131181
balance             0.129106
tenure              0.080836
active_member       0.036143
country_Germany     0.031089
gender_Male         0.021308
credit_card         0.018019
country_Spain       0.015108
has_zero_balance    0.014374
dtype: float64

In [33]:
from sklearn.inspection import permutation_importance

perm_result = permutation_importance(rf_model, X_test, y_test, n_repeats=10, random_state=42)

perm_importances = pd.Series(perm_result.importances_mean, index=X_test.columns).sort_values(ascending=False)
perm_importances

age                 0.06600
products_number     0.05865
active_member       0.02475
country_Germany     0.01500
balance             0.01115
tenure              0.00435
estimated_salary    0.00275
country_Spain       0.00150
gender_Male         0.00135
has_zero_balance    0.00115
credit_score        0.00105
credit_card        -0.00115
dtype: float64

In [34]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(random_state=42, scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum())
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.90      0.88      0.89      1593
           1       0.56      0.61      0.59       407

    accuracy                           0.82      2000
   macro avg       0.73      0.75      0.74      2000
weighted avg       0.83      0.82      0.83      2000



In [35]:
xgb_importances = pd.Series(xgb_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
xgb_importances

products_number     0.221660
has_zero_balance    0.197684
active_member       0.124897
age                 0.096463
country_Germany     0.084747
gender_Male         0.050043
country_Spain       0.047830
balance             0.047537
credit_score        0.035894
estimated_salary    0.032336
credit_card         0.030614
tenure              0.030297
dtype: float32

In [36]:
import pandas as pd
import sys
sys.path.append('..')

from src.preprocessing import run_pipeline

df = pd.read_csv("../data/raw_churn_data.csv")
X_train, X_test, y_train, y_test = run_pipeline(df)
X_train.columns.tolist()

['credit_score',
 'age',
 'tenure',
 'balance',
 'products_number',
 'credit_card',
 'active_member',
 'estimated_salary',
 'has_zero_balance',
 'country_Germany',
 'country_Spain',
 'gender_Male']

In [37]:
from src.preprocessing import save_processed_data

save_processed_data(X_train, X_test, y_train, y_test)

Saved processed data to ../data


In [38]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_balanced_v2 = LogisticRegression(max_iter=1000, class_weight='balanced')
model_balanced_v2.fit(X_train_scaled, y_train)

y_pred_v2 = model_balanced_v2.predict(X_test_scaled)
print(classification_report(y_test, y_pred_v2))

              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1593
           1       0.39      0.71      0.50       407

    accuracy                           0.71      2000
   macro avg       0.65      0.71      0.65      2000
weighted avg       0.80      0.71      0.74      2000



d:\CustomerIQ\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


In [39]:
from sklearn.ensemble import RandomForestClassifier

rf_model_v2 = RandomForestClassifier(random_state=42, class_weight='balanced')
rf_model_v2.fit(X_train, y_train)

y_pred_rf_v2 = rf_model_v2.predict(X_test)
print(classification_report(y_test, y_pred_rf_v2))

              precision    recall  f1-score   support

           0       0.87      0.97      0.92      1593
           1       0.80      0.45      0.57       407

    accuracy                           0.86      2000
   macro avg       0.84      0.71      0.75      2000
weighted avg       0.86      0.86      0.85      2000



In [40]:
from xgboost import XGBClassifier

xgb_model_v2 = XGBClassifier(random_state=42, scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum())
xgb_model_v2.fit(X_train, y_train)

y_pred_xgb_v2 = xgb_model_v2.predict(X_test)
print(classification_report(y_test, y_pred_xgb_v2))

              precision    recall  f1-score   support

           0       0.90      0.88      0.89      1593
           1       0.56      0.61      0.59       407

    accuracy                           0.82      2000
   macro avg       0.73      0.75      0.74      2000
weighted avg       0.83      0.82      0.83      2000



## Feature selection experiment: dropping weak features

Tested dropping `estimated_salary`, `credit_score`, `tenure`, `credit_card`, all four showed
weak/no relationship with churn across EDA (flat groupby averages, near-zero correlation) and
Random Forest permutation importance.

**Result:** dropping them left logistic regression essentially unchanged, but cost both
Random Forest and XGBoost a small, consistent amount of precision (Random Forest: 0.80 → 0.60
precision; XGBoost F1: 0.59 → 0.58). 

**Conclusion:** reverted, kept all 12 original features. Individual per-feature weakness
(via correlation or permutation importance, which isolate one feature at a time) doesn't
guarantee a feature is safe to drop, tree-based models can extract real signal from
*interactions* between features that look individually weak. This is a real limitation of
single-feature importance analysis worth remembering for future feature selection decisions.

In [41]:
import numpy as np
from sklearn.metrics import confusion_matrix

def find_best_threshold(model, X_test, y_test, fn_cost=600, fp_cost=45):
    """Find the probability threshold that minimizes total business cost."""
    probs = model.predict_proba(X_test)[:, 1]
    
    results = []
    for threshold in np.arange(0.05, 0.95, 0.05):
        preds = (probs >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
        total_cost = (fn * fn_cost) + (fp * fp_cost)
        results.append({'threshold': threshold, 'fn': fn, 'fp': fp, 'total_cost': total_cost})
    
    results_df = pd.DataFrame(results)
    return results_df.sort_values('total_cost')

In [42]:
find_best_threshold(xgb_model, X_test, y_test)

,threshold,fn,fp,total_cost
1,0.10,46,727,60315
0,0.05,30,958,61110
2,0.15,67,580,66300
3,0.20,78,473,68085
4,0.25,85,398,68910
5,0.30,108,344,80280
6,0.35,128,298,90210
7,0.40,140,257,95565
8,0.45,150,233,100485
9,0.50,157,194,102930


In [43]:
def find_best_threshold_with_capacity(model, X_test, y_test, fn_cost=600, fp_cost=45, max_flag_rate=0.15):
    """Find the cost-minimizing threshold, constrained to flagging at most max_flag_rate of customers."""
    probs = model.predict_proba(X_test)[:, 1]
    
    results = []
    for threshold in np.arange(0.05, 0.95, 0.05):
        preds = (probs >= threshold).astype(int)
        flag_rate = preds.mean()
        tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
        total_cost = (fn * fn_cost) + (fp * fp_cost)
        results.append({'threshold': threshold, 'flag_rate': flag_rate, 'fn': fn, 'fp': fp, 'total_cost': total_cost})
    
    results_df = pd.DataFrame(results)
    within_capacity = results_df[results_df['flag_rate'] <= max_flag_rate]
    return within_capacity.sort_values('total_cost')

In [44]:
find_best_threshold_with_capacity(xgb_model, X_test, y_test)

,threshold,flag_rate,fn,fp,total_cost
13,0.70,0.1480,206,95,127875
14,0.75,0.1320,221,78,136110
15,0.80,0.1130,240,59,146655
16,0.85,0.0925,259,37,157065
17,0.90,0.0760,278,23,167835


In [45]:
find_best_threshold(rf_model, X_test, y_test)

,threshold,fn,fp,total_cost
0,0.05,15,1050,56250
1,0.10,41,741,57945
2,0.15,73,496,66120
3,0.20,89,376,70320
4,0.25,112,274,79530
5,0.30,143,193,94485
6,0.35,165,139,105255
7,0.40,182,107,114015
8,0.45,203,78,125310
9,0.50,221,51,134895


In [46]:
find_best_threshold_with_capacity(rf_model, X_test, y_test)

,threshold,flag_rate,fn,fp,total_cost
8,0.45,0.1410,203,78,125310
9,0.50,0.1185,221,51,134895
10,0.55,0.1005,239,33,144885
11,0.60,0.0810,267,22,161190
12,0.65,0.0680,288,17,173565
13,0.70,0.0570,305,12,183540
14,0.75,0.0445,325,7,195315
15,0.80,0.0340,346,7,207915
16,0.85,0.0155,378,2,226890
17,0.90,0.0065,394,0,236400


In [47]:
find_best_threshold(model_balanced, X_test, y_test)

d:\CustomerIQ\venv\Lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


,threshold,fn,fp,total_cost
7,0.40,0,1587,71415
14,0.75,0,1587,71415
13,0.70,0,1587,71415
12,0.65,0,1587,71415
11,0.60,0,1587,71415
10,0.55,0,1587,71415
9,0.50,0,1587,71415
8,0.45,0,1587,71415
15,0.80,0,1587,71415
16,0.85,0,1587,71415


In [48]:
find_best_threshold_with_capacity(model_balanced, X_test, y_test)

d:\CustomerIQ\venv\Lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


,threshold,flag_rate,fn,fp,total_cost


In [49]:
find_best_threshold(model_balanced, X_test_scaled, y_test)

,threshold,fn,fp,total_cost
4,0.25,20,1155,63975
3,0.20,9,1312,64440
6,0.35,47,849,66405
5,0.30,38,1000,67800
2,0.15,5,1446,68070
7,0.40,64,709,70305
1,0.10,1,1552,70440
0,0.05,1,1591,72195
8,0.45,93,567,81315
9,0.50,120,451,92295


In [50]:
find_best_threshold_with_capacity(model_balanced, X_test_scaled, y_test)

,threshold,flag_rate,fn,fp,total_cost
13,0.70,0.1490,249,140,155700
14,0.75,0.1015,302,98,185610
15,0.80,0.0660,333,58,202410
16,0.85,0.0400,359,32,216840
17,0.90,0.0155,387,11,232695


In [51]:
probs_check = model_balanced.predict_proba(X_test_scaled)[:, 1]
probs_check.min(), probs_check.max(), probs_check.mean()

(np.float64(0.040782862123962836),
 np.float64(0.9633163658108158),
 np.float64(0.43827105282048734))

In [52]:
for threshold in np.arange(0.05, 0.95, 0.05):
    preds_check = (probs_check >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds_check).ravel()
    total_cost = (fn * 600) + (fp * 45)
    print(f"{threshold:.2f}: fn={fn}, fp={fp}, cost={total_cost}")

0.05: fn=1, fp=1591, cost=72195
0.10: fn=1, fp=1552, cost=70440
0.15: fn=5, fp=1446, cost=68070
0.20: fn=9, fp=1312, cost=64440
0.25: fn=20, fp=1155, cost=63975
0.30: fn=38, fp=1000, cost=67800
0.35: fn=47, fp=849, cost=66405
0.40: fn=64, fp=709, cost=70305
0.45: fn=93, fp=567, cost=81315
0.50: fn=120, fp=451, cost=92295
0.55: fn=148, fp=352, cost=104640
0.60: fn=182, fp=265, cost=121125
0.65: fn=212, fp=195, cost=135975
0.70: fn=249, fp=140, cost=155700
0.75: fn=302, fp=98, cost=185610
0.80: fn=333, fp=58, cost=202410
0.85: fn=359, fp=32, cost=216840
0.90: fn=387, fp=11, cost=232695


In [53]:
for threshold in np.arange(0.05, 0.95, 0.05):
    preds_check = (probs_check >= threshold).astype(int)
    flag_rate = preds_check.mean()
    if flag_rate <= 0.15:
        tn, fp, fn, tp = confusion_matrix(y_test, preds_check).ravel()
        total_cost = (fn * 600) + (fp * 45)
        print(f"{threshold:.2f}: flag_rate={flag_rate:.4f}, fn={fn}, fp={fp}, cost={total_cost}")

0.70: flag_rate=0.1490, fn=249, fp=140, cost=155700
0.75: flag_rate=0.1015, fn=302, fp=98, cost=185610
0.80: flag_rate=0.0660, fn=333, fp=58, cost=202410
0.85: flag_rate=0.0400, fn=359, fp=32, cost=216840
0.90: flag_rate=0.0155, fn=387, fp=11, cost=232695


## Cost-based threshold selection

Assumed business costs: **£600 per missed churner** (false negative - lost customer,
lost future revenue, replacement acquisition cost) and **£45 per false alarm**
(false positive - wasted retention outreach), roughly a 13:1 ratio reflecting that
losing a customer is far more expensive than an unnecessary retention contact.

Tested thresholds from 0.05-0.90 for all three models, both **unconstrained** (minimize
cost regardless of how many customers get flagged) and **capacity-constrained**
(assuming a retention team can realistically contact at most 15% of customers at once).

| Model | Unconstrained optimum | Cost | Capacity-constrained optimum | Cost |
|---|---|---|---|---|
| Random Forest | 0.05 | £56,250 | 0.45 | £125,310 |
| XGBoost | 0.10 | £60,315 | 0.70 | £127,875 |
| Logistic Regression | 0.25 | £63,975 | 0.70 | £155,700 |

**Conclusion:** Random Forest wins under both scenarios, despite XGBoost having the better
F1-score at the default 0.5 threshold. This shows that "best model" depends heavily on how
you evaluate it, default-threshold classification metrics gave a different answer than
cost-optimized, capacity-aware evaluation.

**Final model chosen for production: Random Forest, threshold 0.45** (the capacity-constrained
optimum, since flagging 60%+ of customers unconstrained isn't operationally realistic).

**Limitation worth noting:** the unconstrained optimum pushes toward very low thresholds,
flagging a large share of customers, mathematically optimal per the cost assumptions, but
not deployable without a capacity constraint reflecting real retention team bandwidth.